In [2]:
import os
import xml.etree.ElementTree as ET
import gpxpy
import numpy as np
import pandas as pd
import numpy as np

NS = {
    "gpx": "http://www.topografix.com/GPX/1/0"
}
def load_gpx(filepath):

    tree = ET.parse(filepath)
    root = tree.getroot()

    coords = []

    for pt in root.findall(".//gpx:trkpt", NS):

        lat = float(pt.attrib["lat"])
        lon = float(pt.attrib["lon"])

        coords.append((lat, lon))

    return np.array(coords)


def load_gpx_time(gpx_file):

    tree = ET.parse(gpx_file)
    root = tree.getroot()

    points = []

    for pt in root.findall(".//gpx:trkpt", NS):

        lat = float(pt.attrib["lat"])
        lon = float(pt.attrib["lon"])

        time_elem = pt.find("gpx:time", NS)

        if time_elem is None:
            continue

        points.append({
            "lat": lat,
            "lon": lon,
            "datetime": pd.to_datetime(time_elem.text)
        })

    return np.array(points)

In [3]:
import os
import xml.etree.ElementTree as ET
import numpy as np
from pyproj import Transformer
from sklearn.cluster import DBSCAN

GPX_FOLDER = "londres_traces_walking"

transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:27700",
    always_xy=True
)

NS = {
    "gpx": "http://www.topografix.com/GPX/1/0"
}

features = []
files_kept = []

for file in os.listdir(GPX_FOLDER):

    if not file.endswith(".gpx"):
        continue

    path = os.path.join(
        GPX_FOLDER,
        file
    )

    try:

        tree = ET.parse(path)
        root = tree.getroot()

        pts = root.findall(
            ".//gpx:trkpt",
            NS
        )

        if len(pts) < 2:
            continue

        start = pts[0]
        end = pts[-1]

        start_lat = float(
            start.attrib["lat"]
        )

        start_lon = float(
            start.attrib["lon"]
        )

        end_lat = float(
            end.attrib["lat"]
        )

        end_lon = float(
            end.attrib["lon"]
        )

        sx, sy = transformer.transform(
            start_lon,
            start_lat
        )

        ex, ey = transformer.transform(
            end_lon,
            end_lat
        )

        p1 = (sx, sy)
        p2 = (ex, ey)

        # symmetric OD
        if p1 > p2:
            p1, p2 = p2, p1

        features.append([
            p1[0],
            p1[1],
            p2[0],
            p2[1]
        ])

        files_kept.append(file)

    except Exception:
        continue

X = np.array(features)

print(
    "Trajectories:",
    len(X)
)

Trajectories: 8549


In [29]:
clusterer = DBSCAN(
    eps=50,
    min_samples=5
)

labels = clusterer.fit_predict(X)

In [30]:
from collections import Counter

sizes = Counter(labels)

for c, n in sizes.most_common():

    if c == -1:
        continue

    print(
        f"Cluster {c}: {n}"
    )

Cluster 9: 116
Cluster 23: 99
Cluster 29: 48
Cluster 18: 41
Cluster 6: 37
Cluster 22: 36
Cluster 20: 32
Cluster 10: 30
Cluster 2: 28
Cluster 32: 28
Cluster 39: 28
Cluster 14: 22
Cluster 16: 18
Cluster 24: 17
Cluster 58: 16
Cluster 68: 16
Cluster 28: 15
Cluster 47: 15
Cluster 64: 14
Cluster 0: 13
Cluster 30: 12
Cluster 41: 12
Cluster 44: 12
Cluster 102: 12
Cluster 48: 11
Cluster 61: 11
Cluster 11: 10
Cluster 19: 10
Cluster 26: 10
Cluster 31: 10
Cluster 53: 10
Cluster 99: 10
Cluster 5: 9
Cluster 50: 9
Cluster 70: 9
Cluster 81: 9
Cluster 86: 9
Cluster 7: 8
Cluster 37: 8
Cluster 38: 8
Cluster 55: 8
Cluster 57: 8
Cluster 62: 8
Cluster 1: 7
Cluster 4: 7
Cluster 15: 7
Cluster 21: 7
Cluster 33: 7
Cluster 46: 7
Cluster 69: 7
Cluster 88: 7
Cluster 92: 7
Cluster 95: 7
Cluster 93: 7
Cluster 106: 7
Cluster 110: 7
Cluster 113: 7
Cluster 3: 6
Cluster 8: 6
Cluster 25: 6
Cluster 27: 6
Cluster 35: 6
Cluster 36: 6
Cluster 40: 6
Cluster 42: 6
Cluster 45: 6
Cluster 101: 6
Cluster 74: 6
Cluster 52: 6
Cluste

In [31]:
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

print("Clusters:", n_clusters)

Clusters: 116


In [32]:
pd.Series(labels).value_counts()

-1      7242
 9       116
 23       99
 29       48
 18       41
        ... 
 105       5
 107       5
 108       5
 109       5
 112       5
Name: count, Length: 117, dtype: int64

In [35]:
import folium

cluster_id = 12
inverse_transformer = Transformer.from_crs(
    "EPSG:27700",
    "EPSG:4326",
    always_xy=True
)

# =====================================================
# CLUSTER INDICES
# =====================================================

idxs = np.where(labels == cluster_id)[0]

# =====================================================
# OD CENTER
# =====================================================

origin_x = X[idxs, 0]
origin_y = X[idxs, 1]

dest_x = X[idxs, 2]
dest_y = X[idxs, 3]

origin_center_x = origin_x.mean()
origin_center_y = origin_y.mean()

dest_center_x = dest_x.mean()
dest_center_y = dest_y.mean()

origin_lon, origin_lat = inverse_transformer.transform(
    origin_center_x,
    origin_center_y
)

dest_lon, dest_lat = inverse_transformer.transform(
    dest_center_x,
    dest_center_y
)

# =====================================================
# MAP
# =====================================================

m = folium.Map(
    location=[
        (origin_lat + dest_lat) / 2,
        (origin_lon + dest_lon) / 2
    ],
    zoom_start=13
)

# =====================================================
# TRAJECTORIES
# =====================================================

for idx in idxs:

    path = os.path.join(
        GPX_FOLDER,
        files_kept[idx]
    )

    tree = ET.parse(path)
    root = tree.getroot()

    pts = root.findall(
        ".//gpx:trkpt",
        NS
    )

    coords = [
        (
            float(p.attrib["lat"]),
            float(p.attrib["lon"])
        )
        for p in pts
    ]

    folium.PolyLine(
        coords,
        weight=2,
        opacity=0.3
    ).add_to(m)

# =====================================================
# ORIGINS / DESTINATIONS
# =====================================================

for idx in idxs:

    lon1, lat1 = inverse_transformer.transform(
        X[idx, 0],
        X[idx, 1]
    )

    lon2, lat2 = inverse_transformer.transform(
        X[idx, 2],
        X[idx, 3]
    )

    folium.CircleMarker(
        [lat1, lon1],
        radius=3,
        color="green",
        fill=True
    ).add_to(m)

    folium.CircleMarker(
        [lat2, lon2],
        radius=3,
        color="red",
        fill=True
    ).add_to(m)

# =====================================================
# CLUSTER CENTERS
# =====================================================

folium.Marker(
    [origin_lat, origin_lon],
    popup=f"Origin center ({len(idxs)} traces)",
    icon=folium.Icon(color="green")
).add_to(m)

folium.Marker(
    [dest_lat, dest_lon],
    popup="Destination center",
    icon=folium.Icon(color="red")
).add_to(m)

folium.PolyLine(
    [
        (origin_lat, origin_lon),
        (dest_lat, dest_lon)
    ],
    color="black",
    weight=5,
    opacity=0.8
).add_to(m)
legend_html = """
<div style="
position: fixed;
bottom: 50px;
left: 50px;
width: 220px;
height: 140px;
background-color: white;
border:2px solid grey;
z-index:9999;
font-size:14px;
padding:10px;
">

<b>OD Cluster Legend</b><br><br>

<span style="color:green;">●</span>
Origins<br>

<span style="color:red;">●</span>
Destinations<br>

<span style="color:black;">━</span>
OD center connection<br>

<span style="color:blue;">━</span>
Trajectories

</div>
"""
m.get_root().html.add_child(
    folium.Element(legend_html)
)
m